In [ ]:
%pip install claude-agent-sdk python-dotenv

import os
from dotenv import load_dotenv
load_dotenv("/Users/robertward/Documents/GitHub/chorus/.env")

from claude_agent_sdk import (
    query, ClaudeAgentOptions,
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
    tool, create_sdk_mcp_server,
)

# MCP server config — starts mcp_rag_server.py as a subprocess over stdio
MCP_SERVERS = {
    "chorus-rag": {
        "command": "/Users/robertward/Documents/GitHub/chorus/.venv/bin/python",
        "args": ["/Users/robertward/Documents/GitHub/chorus/scripts/mcp_rag_server.py"],
        "env": {"ANTHROPIC_API_KEY": os.environ["ANTHROPIC_API_KEY"]},
    }
}

CHORUS_SYSTEM_PROMPT = (
    "You are Chorus, a playful, knowledgeable and efficient assistant for the "
    "Knowledge Lab at University of Chicago. Your goal, in the vein of Christopher "
    "Alexander, is to help make the lab and its members whole. You operate as pooled "
    "memory for the lab, as a mechanic that helps members solve concrete problems, as "
    "a matchmaker who directs members into social interactions to support collaboration "
    "and culture, and as a muse generating surprising angles on the work being done. "
    "Inspired by the Chorus of Greek theater, you have multiple voices that can be "
    "called as subagents. Use your RAG tools (search_documents, get_context, "
    "list_sources) to ground your answers in the Knowledge Lab's actual work."
)

# Custom calculator tool via @tool decorator + in-process SDK MCP server
@tool("calculator", "Evaluate a math expression for cost calculations.", {"expression": str})
async def calculator(args):
    try:
        result = str(eval(args["expression"]))
    except Exception as e:
        result = f"Error evaluating expression: {e}"
    return {"content": [{"type": "text", "text": result}]}

calc_server = create_sdk_mcp_server(name="calc", tools=[calculator])

print("Setup complete.")

In [ ]:
options = ClaudeAgentOptions(
    system_prompt=CHORUS_SYSTEM_PROMPT,
    mcp_servers={**MCP_SERVERS, "calc": calc_server},
    allowed_tools=[
        "mcp__chorus-rag__search_documents",
        "mcp__chorus-rag__get_context",
        "mcp__chorus-rag__list_sources",
        "mcp__calc__calculator",
    ],
    permission_mode="bypassPermissions",
    max_turns=10,
)

async for message in query(prompt="Tell me about yourself.", options=options):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(block.text)
            elif isinstance(block, ToolUseBlock):
                print(f"  [tool call: {block.name}]")
    elif isinstance(message, ResultMessage):
        print(f"\nResult: {message.result}")
        print(f"Cost: ${message.total_cost_usd:.4f}")
        print(f"Duration: {message.duration_ms / 1000:.1f}s")